# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge: Data Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the [`mlcroissant`](https://mlcroissant.org/) library, a standardized interface for FAIR data described using the Croissant schema.

### Dataset Source
This dataset comes from a Croissant schema at:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}:\n{metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Authors: {[a for a in getattr(metadata, 'author', [])]}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review the available record sets and their fields using their `@id` fields.

**All entities (record sets, fields, columns, etc.) should always be referenced by their `@id`.**

In [ ]:
# Discover available record sets
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s)\n")
for rset in record_sets:
    print(f"RecordSet: @id = {rset['@id']}")
    print(f"  Name     : {rset.get('name', 'No name')}")
    print("  Fields:")
    for field in rset.get('field', []):
        print(f"      - @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")
    print()
# Save the first RecordSet @id
record_set_ids = [rset['@id'] for rset in record_sets]
if record_set_ids:
    first_record_set_id = record_set_ids[0]

## 3. Data Extraction
Let's load the data from the first available record set into a pandas DataFrame for further exploration.

*Replace and reference all entities by their `@id` field as per Croissant conventions.*

If the dataset contains multiple record sets, data will be loaded for each.

In [ ]:
# Load records for each record set
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2))
        print()
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Display again columns from the first record set
if record_set_ids:
    print(f"Columns in {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will apply some data processing steps: filtering, normalization, and grouping. 

**Note:** Specify fields and columns by their `@id`, as per FAIR and Croissant best practices. 

Below, select a numeric field and a grouping field from the `@id` of the relevant columns in the data overview above. Adapt the field choices as appropriate for your recordset.

In [ ]:
# Replace with an actual numeric column @id from your record set
# If unsure, print(dataframes[first_record_set_id].columns) and select a numeric column
# For this notebook, we'll try to select a common regression output column name
df = dataframes[first_record_set_id]
numeric_field_id = None
possible_numeric_ids = [col for col in df.columns if 'log_likelihood' in col.lower() or 'coef' in col.lower() or 'std' in col.lower() or 'value' in col.lower() or 'score' in col.lower() ]
if len(possible_numeric_ids) > 0:
    numeric_field_id = possible_numeric_ids[0]
else:
    numeric_field_id = df.columns[0] if df.shape[1]>0 else None
print(f"Using numeric field @id: {numeric_field_id}")

# Choose a group field @id if available
possible_group_ids = [col for col in df.columns if 'gender' in col.lower() or 'ward' in col.lower() or 'cluster' in col.lower() or 'county' in col.lower()]
if len(possible_group_ids) > 0:
    group_field_id = possible_group_ids[0]
else:
    group_field_id = None
print(f"Using group field @id: {group_field_id}")

# EDA: Filter, normalize, group
if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].quantile(0.75)
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No valid numeric field for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and (if available) differences across the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("Cannot plot: numeric field not found or not numeric.")

## 6. Conclusion

- Loaded and reviewed the dataset structure and metadata from the Croissant schema.
- Explored record sets, fields, and visualized the distribution and groupings of a key numeric variable.
- All dataset entities were referenced by their `@id`, using the standardized approach recommended by the Croissant schema and `mlcroissant` library.

Further analysis can focus on model interpretability, bias assessment, or linking socio-demographic predictors to knowledge adoption in rangeland management.